Для удобства тестирования

In [ ]:
# Импорты перезагружаются каждый раз при запуске любой ячейки для удобства отладки govsim
%load_ext autoreload
%autoreload 2

In [ ]:
from govsim import simulation

In [ ]:
simulation.run_simulation()

In [11]:
# gov_agent_linear.py
import random
import time
from typing import Dict, Any, List, Optional, Set, Tuple
import re
import json
import numpy as np # Для расчета MSE/MSU
import traceback

# Импортируем интерфейсы и утилиты
from govsim.utils.interfaces import BaseGovernmentAgent, Policy, PolicyDescriptor, BaseEconomicSystem
from govsim.utils.policy_utils import validate_and_compile_policy_expression, PolicyValidationError

try:
    from govsim.utils.gemini_utils import create_agent, BaseAgent as GeminiBaseAgent
except ImportError as e:
    print(f"ПРЕДУПРЕЖДЕНИЕ из gov_agent_linear.py: Не удалось импортировать .gemini_utils. Ошибка: {e}. IntelligentLLMAgent может не работать.")
    GeminiBaseAgent = None
    import traceback
    traceback.print_exc()

In [30]:
def _parse_llm_response(response_text: str) -> Optional[Dict[str, str]]:
    # Пытаемся извлечь JSON блок из ответа LLM
    # print(f"DEBUG LLM Raw Response:\n---\n{response_text}\n---")
    match = re.search(r'```json\s*(\{.*?\})\s*```', response_text, re.DOTALL)
    json_str = None
    if match:
        json_str = match.group(1)
    else:
        # Если нет ```json```, пытаемся найти JSON объект просто в тексте
        match_plain_json = re.search(r'(\{.*?\})', response_text, re.DOTALL)
        if match_plain_json:
            json_str = match_plain_json.group(1)
        # Если и простого JSON нет, попробуем найти его без начальной { и конечной }
        # Это нужно из-за моделей типа Лаконичного Агента
        else:
            match_inner_json = re.search(r'"policy_type_id":.*"reasoning":\s*".*?"', response_text, re.DOTALL)
            if match_inner_json:
                json_str = "{" + match_inner_json.group(0) + "}"


    if not json_str:
        print("ПРЕДУПРЕЖДЕНИЕ LLM Agent: Не найден JSON в ответе LLM.")
        # Попробуем найти хотя бы value_expression, если стиль laconic
        if self.llm_style == 'laconic':
            # Ищем просто строку в кавычках или без кавычек после ключевого слова
                expr_match = re.search(r'(?:value_expression["\']?\s*[:=]?\s*["\']?)(.*?)(?:["\']?\s*,?\s*reasoning|$)', response_text, re.IGNORECASE | re.DOTALL)
                if expr_match:
                    expr = expr_match.group(1).strip().replace('`','').replace('\n',' ')
                    # Пытаемся угадать policy_type_id (например, единственный доступный)
                    # Это очень ненадежно, нужно указывать в промпте для Laconic явно!
                    # Здесь просто заглушка:
                    guessed_policy_type = "set_control_input" # Или взять из дескрипторов, если он один
                    print(f"ПРЕДУПРЕЖДЕНИЕ: JSON не найден, но в Laconic стиле извлечено выражение: '{expr}'. Угаданный тип: '{guessed_policy_type}'")
                    return {
                        "policy_type_id": guessed_policy_type,
                        "value_expression": expr,
                        "reasoning": "(не извлечено из laconic ответа)"
                    }

        return None # Если не laconic или не нашли выражение

    try:
        # Очистка JSON строки (замена кавычек, удаление комментариев, хвостовых запятых)
        #cleaned_json_str = json_str.replace("'", '"')
        cleaned_json_str = json_str
        # Удаляем однострочные комментарии // и /* ... */ (если вдруг LLM их добавит)
        cleaned_json_str = re.sub(r"//.*?\n", "\n", cleaned_json_str)
        cleaned_json_str = re.sub(r"/\*.*?\*/", "", cleaned_json_str, flags=re.DOTALL)
        # Удаление хвостовых запятых перед } или ]
        cleaned_json_str = re.sub(r',\s*([\}\]])', r'\1', cleaned_json_str)

        parsed_data = json.loads(cleaned_json_str)

        if isinstance(parsed_data, dict) and \
            "policy_type_id" in parsed_data and \
            "value_expression" in parsed_data and \
            "reasoning" in parsed_data:
            # Простая проверка типов
            if not isinstance(parsed_data["policy_type_id"], str) or \
                not isinstance(parsed_data["value_expression"], str) or \
                not isinstance(parsed_data["reasoning"], str):
                print("ПРЕДУПРЕЖДЕНИЕ LLM Agent: Некорректные типы данных в JSON от LLM.")
                return None
            # Дополнительно чистим value_expression от возможных артефактов
            value_expr_clean = parsed_data["value_expression"].strip().replace('`','').replace('\n',' ')

            return {
                "policy_type_id": parsed_data["policy_type_id"].strip(),
                "value_expression": value_expr_clean,
                "reasoning": parsed_data["reasoning"].strip()
            }
        else:
            print("ПРЕДУПРЕЖДЕНИЕ LLM Agent: JSON от LLM не содержит всех необходимых ключей (policy_type_id, value_expression, reasoning).")
            return None
    except json.JSONDecodeError as e:
        print(f"ОШИБКА LLM Agent: Не удалось распарсить JSON из ответа LLM: {e}\nСтрока JSON: '{json_str}'")
        return None

In [ ]:
_parse_llm_response('''{{
  "policy_type_id": "set_control_input",
  "value_expression": "np.clip(-1.9 * (current_x - target_x), -2.0, 2.0)",
  "reasoning": "1. **Краткая оценка текущей ситуации:** Текущее состояние `current_x` (0.0533) немного отклонено от целевого значения `target_x` (0.000) в положительную сторону. Показатели производительности (MSE=0.0125, MSU=0.0179) находятся на разумном уровне, но есть возможность для дальнейшего уменьшения MSE. \n2. **Ожидаемый эффект от `u_k` на `x_(k+1)`:** Выбранное управляющее воздействие `u_k` основано на принципе 'deadbeat control', стремясь привести `x_(k+1)` максимально близко к `target_x` за один шаг, игнорируя стохастический шум. Формула `u_k = (target_x - param_A * current_x - param_C) / param_B` с подстановкой параметров `param_A=0.95`, `param_B=0.5`, `param_C=0`, `target_x=0` упрощается до `u_k = -1.9 * current_x`. Для `current_x = 0.0533` это приводит к `u_k` приблизительно -0.101. Это отрицательное управляющее воздействие будет активно уменьшать текущее положительное отклонение `current_x`, направляя `x_(k+1)` к нулю. \n3. **Как решение балансирует цели MSE и MSU:** Данная стратегия является агрессивной по отношению к минимизации MSE, поскольку она напрямую нацелена на достижение `target_x` на следующем шаге. Однако, поскольку текущее отклонение `current_x` невелико, вычисленное значение `u_k` также невелико (около -0.101), что способствует поддержанию низкого MSU. В случае больших отклонений `current_x`, значение `u_k` будет пропорционально увеличиваться, но оно будет автоматически ограничено диапазоном `(-2.0, 2.0)` системой, что предотвратит чрезмерно высокие значения MSU и обеспечит стабильность управления. Таким образом, достигается сильный приоритет на низкий MSE при разумном контроле MSU."
  ''')